In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler,OneHotEncoder,LabelEncoder
from sklearn.impute import SimpleImputer
import seaborn as sns

In [2]:
penguins = sns.load_dataset('penguins')
penguins

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,NaN,NaN,NaN,NaN,NaN
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female


In [3]:
penguins.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    object 
 1   island             344 non-null    object 
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    object 
dtypes: float64(4), object(3)
memory usage: 18.9+ KB


In [4]:
penguins.isnull().sum()

species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64

In [ ]:
num_cols=penguins.select_dtypes('number').columns  #It gives you the column names of the DataFrame as an Index object (basically a list of column names).
print("numericals columns:",num_cols.tolist())

numericals columns: ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']


In [ ]:
cat_cols=penguins.select_dtypes('object').columns
''' 
select_dtypes('object') or include = 'object' or exclude = 'number' '''
print("categoricals:",cat_cols.tolist())

categoricals: ['species', 'island', 'sex']


In [ ]:
num_imp=SimpleImputer(strategy='mean')#default strategy is 'mean'
'''
SimpleImputer is a class from sklearn.impute. It fills (imputes) missing values.
SimpleImputer(...)
→ Creates an object, like how you initialize a model.
strategy='mean'
→ This tells the imputer how to fill missing values.'''
cat_imp=SimpleImputer(strategy='most_frequent')

In [ ]:
penguins[num_cols]=num_imp.fit_transform(penguins[num_cols])
'''
fit():
Calculates the mean of each numerical column.
transform():
Replaces all NaN values with those means.
OUTPUT:
A new NumPy array with the NaN values filled in.
Important point:
👉 This does not modify the original DataFrame by itself but here its modifiying the actual dataset cause we've asked it to do so
'''
penguins.isnull().sum()
# 

species               0
island                0
bill_length_mm        0
bill_depth_mm         0
flipper_length_mm     0
body_mass_g           0
sex                  11
dtype: int64

In [ ]:
penguins[['sex']]=cat_imp.fit_transform(penguins[['sex']])#two brackets to keep it as  dataframe
'''  we're not using cat_cols in place of 'sex' cause othe 2 doesn't have null values
fit():
Looks at the column ‘sex’
Finds the most frequent value (e.g., ‘male’)
transform():
Replaces all NaN values with that most frequent category.'''
penguins.isnull().sum()

species              0
island               0
bill_length_mm       0
bill_depth_mm        0
flipper_length_mm    0
body_mass_g          0
sex                  0
dtype: int64

ENCODING categorical values

in dataset we divide cols into features and lable : lable data is the data which we want ML model to pridict and feature cols are the columns which decides the value of lable col
- train_test_split : we divide the data 80% for training and 20% for testing 
- OneHotEncoder : for encoding feature columns
- LableEndoder : for encoding lable column

In [ ]:
#we want Ml model to predict gender so we made 'sex' col as lable col and we're using LableEncoder on it
sex_enc=LabelEncoder()
''' 
You’re creating an object of the LabelEncoder class from sklearn.preprocessing.
This encoder converts categorical text labels → numerical labels.
It doesn’t impute, doesn’t fill missing values — it only encodes categories.
“Take each unique category and map it to an integer.” might produce: female → 0   male → 1
'''
penguins['sex']=sex_enc.fit_transform(penguins['sex'])


In [11]:
print(penguins['sex'])

0      1
1      0
2      0
3      1
4      0
      ..
339    1
340    0
341    1
342    0
343    1
Name: sex, Length: 344, dtype: int64


In [23]:
penguins.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.10000,18.70000,181.000000,3750.000000,1
1,Adelie,Torgersen,39.50000,17.40000,186.000000,3800.000000,0
2,Adelie,Torgersen,40.30000,18.00000,195.000000,3250.000000,0
3,Adelie,Torgersen,43.92193,17.15117,200.915205,4201.754386,1
4,Adelie,Torgersen,36.70000,19.30000,193.000000,3450.000000,0


dropping the first column hels avoid the dummy variable trap.
when we perform one hot encoding, anew column is created for every category.if you encode n categories , you will get n columns. 
but on eof those columns can always be predicted from the combination of the other columns.
 this is known as the dummy variable trap, and it creates multicollinearity (which is a problem for linear models)

 Multicollinearity is a is a statistical phenomeon that occurs when:
 one or more independent variables (features)are strongly correlated with other independent variable.  

In [ ]:
cat_enc=OneHotEncoder(drop='first') #we want to encode 
dummy_cols=cat_enc.fit_transform(penguins[cat_cols]).toarray()  #we're converting this to array cause array is made up into df in pandas
dummy_df=pd.DataFrame(dummy_cols) #converting dummy_cols array into dataframe
dummy_df

,0,1,2,3,4
0,0.0,0.0,0.0,1.0,1.0
1,0.0,0.0,0.0,1.0,0.0
2,0.0,0.0,0.0,1.0,0.0
3,0.0,0.0,0.0,1.0,1.0
4,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...
339,0.0,1.0,0.0,0.0,1.0
340,0.0,1.0,0.0,0.0,0.0
341,0.0,1.0,0.0,0.0,1.0
342,0.0,1.0,0.0,0.0,0.0


In [13]:
penguins['species'].value_counts()

species
Adelie       152
Gentoo       124
Chinstrap     68
Name: count, dtype: int64

In [14]:
dummy_df

,0,1,2,3,4
0,0.0,0.0,0.0,1.0,1.0
1,0.0,0.0,0.0,1.0,0.0
2,0.0,0.0,0.0,1.0,0.0
3,0.0,0.0,0.0,1.0,1.0
4,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...
339,0.0,1.0,0.0,0.0,1.0
340,0.0,1.0,0.0,0.0,0.0
341,0.0,1.0,0.0,0.0,1.0
342,0.0,1.0,0.0,0.0,0.0


In [ ]:
clean_df = pd.concat([penguins,dummy_df],axis=1).drop(columns=cat_cols)
clean_df
''' 
pd.concat() is used to combine DataFrames.
[penguins, dummy_df] → we’re combining the original penguins DataFrame with the dummy variables DataFrame (dummy_df).
axis=1 → means we are concatenating horizontally, i.e., adding columns side by side (not stacking rows).
Result → a DataFrame with all original columns + all dummy columns.
.drop(columns=cat_cols)
After adding the dummy variables, you usually don’t need the original categorical columns anymore (cat_cols).
.drop(columns=cat_cols) removes those original categorical columns from the combined DataFrame.
'''

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,0,1,2,3,4
0,39.10000,18.70000,181.000000,3750.000000,0.0,0.0,0.0,1.0,1.0
1,39.50000,17.40000,186.000000,3800.000000,0.0,0.0,0.0,1.0,0.0
2,40.30000,18.00000,195.000000,3250.000000,0.0,0.0,0.0,1.0,0.0
3,43.92193,17.15117,200.915205,4201.754386,0.0,0.0,0.0,1.0,1.0
4,36.70000,19.30000,193.000000,3450.000000,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...
339,43.92193,17.15117,200.915205,4201.754386,0.0,1.0,0.0,0.0,1.0
340,46.80000,14.30000,215.000000,4850.000000,0.0,1.0,0.0,0.0,0.0
341,50.40000,15.70000,222.000000,5750.000000,0.0,1.0,0.0,0.0,1.0
342,45.20000,14.80000,212.000000,5200.000000,0.0,1.0,0.0,0.0,0.0


In [ ]:
scaler=StandardScaler()
clean_df[num_cols]=scaler.fit_transform(clean_df[num_cols])
clean_df
''' 
StandardScaler()
This comes from sklearn.preprocessing.
Its job: standardize numeric data.
How? By transforming each column so that it has:
Mean = 0
Standard deviation = 1
scaler.fit_transform(clean_df[num_cols])
.fit() → calculates the mean and standard deviation of each numeric column.
.transform() → uses these stats to scale the data.
.fit_transform() → does both in one step.
'''

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,0,1,2,3,4
0,-8.870812e-01,7.877425e-01,-1.422488,-0.565789,0.0,0.0,0.0,1.0,1.0
1,-8.134940e-01,1.265563e-01,-1.065352,-0.503168,0.0,0.0,0.0,1.0,0.0
2,-6.663195e-01,4.317192e-01,-0.422507,-1.192003,0.0,0.0,0.0,1.0,0.0
3,-1.307172e-15,1.806927e-15,0.000000,0.000000,0.0,0.0,0.0,1.0,1.0
4,-1.328605e+00,1.092905e+00,-0.565361,-0.941517,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...
339,-1.307172e-15,1.806927e-15,0.000000,0.000000,0.0,1.0,0.0,0.0,1.0
340,5.294731e-01,-1.450118e+00,1.006038,0.811880,0.0,1.0,0.0,0.0,0.0
341,1.191758e+00,-7.380718e-01,1.506028,1.939064,0.0,1.0,0.0,0.0,1.0
342,2.351241e-01,-1.195816e+00,0.791756,1.250229,0.0,1.0,0.0,0.0,0.0


In [17]:
#pipeline
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [ ]:
df=sns.load_dataset('penguins')
num_cols=df.select_dtypes('number').columns
cat_cols=df.select_dtypes('object').columns

num_pipeline=Pipeline([
    ('imputer',SimpleImputer()),('scaler',StandardScaler())
])
'''
Pipeline lets you chain multiple preprocessing steps into one object.
num_pipeline for numeric columns does:
SimpleImputer() → fills missing values (default: mean for numbers).
StandardScaler() → scales numeric features so mean=0, std=1. 
'''
cat_pipeline=Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('encoder',OneHotEncoder(drop='first'))
])
'''
cat_pipeline for categorical columns does:
SimpleImputer(strategy='most_frequent') → fills missing values with the most frequent category.
OneHotEncoder(drop='first') → converts categorical columns to dummy variables (0/1), and drop='first' avoids multicollinearity by dropping the first category. 
'''

In [ ]:
preprocessor=ColumnTransformer(
    transformers=[('num',num_pipeline,num_cols),('cat',cat_pipeline,cat_cols)
    ]
)
''' ColumnTransformer lets you apply different preprocessing steps to different columns:
'num' → name of the transformer for numeric columns
num_pipeline → the actual steps (like scaling, imputation, etc.)
num_cols → which columns it should apply to
Same logic for cat, except for categorical columns. '''

In [20]:
preprocessor.fit_transform(df)

array([[-0.88708123,  0.78774251, -1.42248782, ...,  0.        ,
         1.        ,  1.        ],
       [-0.81349399,  0.12655633, -1.06535169, ...,  0.        ,
         1.        ,  0.        ],
       [-0.66631952,  0.43171918, -0.42250666, ...,  0.        ,
         1.        ,  0.        ],
       ...,
       [ 1.1917582 , -0.73807176,  1.50602843, ...,  0.        ,
         0.        ,  1.        ],
       [ 0.23512413, -1.19581604,  0.79175618, ...,  0.        ,
         0.        ,  0.        ],
       [ 1.09977416, -0.53462985,  0.8631834 , ...,  0.        ,
         0.        ,  1.        ]], shape=(344, 9))